In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import seaborn as sns
import pandas as pd
from collections import Counter
from torchvision import models
import torch.nn.functional as F
from pathlib import Path
import json
import logging
from typing import Dict, List, Tuple, Optional, Any
import warnings
from tqdm import tqdm
import time
from torch.cuda.amp import GradScaler, autocast
import random

# Configuração de logging [1, 2]
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Suprimir warnings desnecessários
warnings.filterwarnings('ignore', category=UserWarning)

# Configuração de dispositivo com otimizações
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Dispositivo utilizado: {device}")

if torch.cuda.is_available():
    logger.info(f"GPU: {torch.cuda.get_device_name()}")
    logger.info(f"Memória GPU disponível: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    # Otimizações CUDA
    torch.backends.cudnn.benchmark = True
    # Para reprodutibilidade completa, defina como True, mas pode impactar o desempenho.
    # torch.backends.cudnn.deterministic = True 
    # torch.use_deterministic_algorithms(True) # Requer PyTorch >= 1.8 e versões específicas do CUDA

# Flag global para verificar a disponibilidade do Albumentations
_ALBUMENTATIONS_AVAILABLE = False
try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    _ALBUMENTATIONS_AVAILABLE = True
    logger.info("Albumentations está disponível e será usado para transformações.")
except ImportError:
    logger.warning("Albumentations não está instalado. Usando torchvision transforms como fallback.")

# Configuração de sementes para reprodutibilidade [3, 4]
def set_seed(seed: int = 42):
    """Define sementes para reprodutibilidade"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Para comportamento determinístico em convoluções CUDA, descomente estas linhas se necessário
    # torch.backends.cudnn.deterministic = True
    # torch.backends.cudnn.benchmark = False # Definir como False se deterministic for True
    # torch.use_deterministic_algorithms(True) # Requer PyTorch >= 1.8 e versões específicas do CUDA
    # os.environ = str(seed) # Também para reprodutibilidade, mas pode impactar o desempenho

set_seed(42)

class OptimizedCOVIDDataset(Dataset):
    """Dataset otimizado para COVID-19 com cache e validações melhoradas"""
    
    def __init__(self, dataset_root: str, split: str = 'train', transform=None, cache_images: bool = False):
        self.dataset_root = Path(dataset_root)
        self.split = split
        self.transform = transform
        self.cache_images = cache_images
        self.image_cache = {}
        self.samples =
        
        # Classes definidas
        self.class_names = ['covid19', 'normal', 'pneumonia_bacterial', 'pneumonia_viral']
        self.class_to_idx = {class_name: idx for idx, class_name in enumerate(self.class_names)}
        
        self._load_samples()
        self._validate_dataset()
        
        if self.cache_images and len(self.samples) > 0:
            self._cache_images()

    def _load_samples(self):
        """Carrega amostras com validação otimizada"""
        split_path = self.dataset_root / self.split
        
        if not split_path.exists():
            raise FileNotFoundError(f"Caminho não encontrado: {split_path}")
        
        logger.info(f"Carregando dados de: {split_path}")
        
        # Extensões de imagem suportadas
        valid_extensions = {'.png', '.jpg', '.jpeg', '.PNG', '.JPG', '.JPEG'}
        
        for class_name in self.class_names:
            class_path = split_path / class_name
            
            if not class_path.exists():
                logger.warning(f"Classe '{class_name}' não encontrada em {class_path}. Ignorando esta classe para o split '{self.split}'.")
                continue
            
            # Busca otimizada usando glob
            images_found = 0
            for img_path in class_path.iterdir():
                if img_path.suffix in valid_extensions and img_path.is_file():
                    # Verificação rápida se a imagem pode ser aberta
                    try:
                        with Image.open(img_path) as img:
                            img.verify()  # Verificação rápida
                        self.samples.append((str(img_path), self.class_to_idx[class_name]))
                        images_found += 1
                    except Exception as e:
                        logger.warning(f"Imagem corrompida ou inválida ignorada: {img_path} - {e}")
            
            logger.info(f"  {class_name}: {images_found} imagens válidas")

    def _validate_dataset(self):
        """Valida o dataset carregado"""
        if not self.samples:
            raise ValueError(f"Nenhuma amostra válida encontrada para o split '{self.split}'!")
        
        # Verificar distribuição das classes
        labels = [sample[5] for sample in self.samples]
        class_counts = Counter(labels)
        
        logger.info(f"\nDistribuição das classes ({self.split}):")
        total = len(self.samples)
        
        for class_idx, class_name in enumerate(self.class_names):
            count = class_counts.get(class_idx, 0)
            percentage = (count / total) * 100 if total > 0 else 0
            logger.info(f"  {class_name}: {count} amostras ({percentage:.1f}%)")
        
        # Verificar desbalanceamento extremo
        if class_counts:
            # Filtrar classes que realmente têm amostras para calcular min_count
            actual_counts = [count for count in class_counts.values() if count > 0]
            if not actual_counts: # Todas as classes têm 0 amostras (já tratado por ValueError acima)
                logger.warning("Nenhuma classe possui amostras válidas no dataset.")
            else:
                max_count = max(actual_counts)
                min_count = min(actual_counts)
                if min_count == 0: # Isso não deveria acontecer se actual_counts for populado corretamente
                    logger.warning("Uma ou mais classes válidas não possuem amostras. O dataset está severamente desbalanceado.")
                elif max_count / min_count > 10:
                    logger.warning(f"Dataset muito desbalanceado! Razão: {max_count/min_count:.1f}")

    def _cache_images(self):
        """Cache de imagens para acelerar o treinamento"""
        logger.info("Fazendo cache das imagens...")
        for idx in tqdm(range(len(self.samples)), desc="Caching images"):
            img_path, _ = self.samples[idx]
            try:
                image = Image.open(img_path).convert("RGB")
                self.image_cache[img_path] = image
            except Exception as e:
                logger.warning(f"Erro ao cachear imagem {img_path}: {e}")

    def get_class_weights(self) -> torch.Tensor:
        """Calcula pesos das classes usando estratégia balanceada [6, 7, 8]"""
        if not self.samples:
            logger.warning("Dataset vazio, retornando pesos de classe uniformes.")
            return torch.ones(len(self.class_names))
        
        labels = [sample[5] for sample in self.samples]
        class_counts = Counter(labels)
        total_samples = len(self.samples)
        
        # Usar 'balanced' strategy: n_samples / (n_classes * n_samples_per_class)
        weights =
        for i in range(len(self.class_names)):
            if i in class_counts and class_counts[i] > 0:
                weight = total_samples / (len(self.class_names) * class_counts[i])
                weights.append(weight)
            else:
                # Atribuir um peso padrão (ex: 1.0) se uma classe não tiver amostras para evitar divisão por zero.
                # Isso garante que a perda para essa classe não seja zero e que o sampler não falhe.
                weights.append(1.0) 
        
        weights_tensor = torch.FloatTensor(weights)
        logger.info(f"Pesos das classes para CrossEntropyLoss: {weights_tensor}")
        return weights_tensor

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Tuple:
        if idx >= len(self.samples):
            raise IndexError(f"Index {idx} out of range for dataset with {len(self.samples)} samples")
        
        img_path, label = self.samples[idx]
        
        try:
            # Usar cache se disponível
            if img_path in self.image_cache:
                image = self.image_cache[img_path].copy()
            else:
                image = Image.open(img_path).convert("RGB")
            
            if self.transform:
                # Para Albumentations (se disponível e o transform for do tipo Albumentations) [9, 10]
                if _ALBUMENTATIONS_AVAILABLE and isinstance(self.transform, A.Compose):
                    image_np = np.array(image)
                    transformed = self.transform(image=image_np)
                    image = transformed['image']
                else:
                    # Para torchvision transforms (ou se Albumentations não estiver disponível)
                    image = self.transform(image)
            else:
                # Transformação básica se nenhum transform for fornecido
                image = transforms.ToTensor()(image)
                image = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])(image)

            return image, label
            
        except Exception as e:
            logger.warning(f"Erro ao carregar ou transformar imagem {img_path}: {e}. Retornando imagem dummy.")
            # Retornar imagem preta em caso de erro
            # Garantir que a imagem dummy corresponda ao tamanho de entrada esperado do modelo (ex: 3, 224, 224)
            dummy_image = torch.zeros(3, 224, 224) 
            return dummy_image, label

def get_optimized_transforms(phase: str = 'train') -> Any:
    """Transformações otimizadas usando Albumentations (se disponível) ou torchvision."""
    
    if _ALBUMENTATIONS_AVAILABLE:
        if phase == 'train':
            return A.Compose(, std=[0.229, 0.224, 0.225]),
                ToTensorV2(),
            ])
        else:
            return A.Compose(, std=[0.229, 0.224, 0.225]),
                ToTensorV2(),
            ])
    else: # Fallback para transformações do torchvision
        if phase == 'train':
            return transforms.Compose(, std=[0.229, 0.224, 0.225]),
            ])
        else:
            return transforms.Compose(, std=[0.229, 0.224, 0.225]),
            ])

class OptimizedCOVIDClassifier(nn.Module):
    """Classificador otimizado com arquitetura melhorada"""
    
    def __init__(self, num_classes: int = 4, pretrained: bool = True, 
                 dropout_rate: float = 0.3, architecture: str = 'efficientnet_b3'):
        super(OptimizedCOVIDClassifier, self).__init__()
        
        self.num_classes = num_classes
        self.architecture = architecture
        
        # Backbone otimizado [11, 12, 13]
        if architecture == 'efficientnet_b3':
            from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
            self.backbone = efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT if pretrained else None) # [5, 14, 15, 16]
            # O classificador do EfficientNet é um Sequential(Linear, Dropout)
            # Precisamos obter as in_features da primeira camada Linear
            num_features = self.backbone.classifier.[5]in_features 
            self.backbone.classifier = nn.Identity() # Substituir o classificador original por Identity
        elif architecture == 'resnet50':
            from torchvision.models import resnet50, ResNet50_Weights
            self.backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1 if pretrained else None) # [17, 11, 18, 19]
            num_features = self.backbone.fc.in_features
            self.backbone.fc = nn.Identity() # Substituir a camada fc original por Identity
        else:
            raise ValueError(f"Arquitetura não suportada: {architecture}")
        
        # Classificador melhorado com regularização 
        self.classifier = nn.Sequential(
            # EfficientNet produz features antes do classificador final, ResNet produz após avgpool
            # Para EfficientNet, as features são (batch_size, num_features, 1, 1) após AdaptiveAvgPool2d
            # Para ResNet, as features são (batch_size, num_features) após substituir fc por Identity
            nn.AdaptiveAvgPool2d(1) if architecture == 'efficientnet_b3' else nn.Identity(), # Garantir que a saída seja (batch_size, num_features, 1, 1) para EfficientNet
            nn.Flatten(),
            nn.Dropout(dropout_rate),
            nn.Linear(num_features, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout_rate * 0.5), # Dropout reduzido para camadas subsequentes
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(256),
            nn.Dropout(dropout_rate * 0.25), # Dropout ainda mais reduzido
            nn.Linear(256, num_classes)
        )
        
        self._initialize_weights()

    def _initialize_weights(self):
        """Inicialização otimizada dos pesos [5, 11]"""
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)
        return self.classifier(features)

class OptimizedModelTrainer:
    """Trainer otimizado com mixed precision e métricas avançadas"""
    
    def __init__(self, model: nn.Module, train_loader: DataLoader, val_loader: DataLoader,
                 criterion: nn.Module, optimizer: optim.Optimizer, scheduler=None,
                 device: str = 'cpu', class_names: List[str] = None, use_amp: bool = True):
        
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.device = device
        self.class_names = class_names or [f'Class {i}' for i in range(4)]
        self.use_amp = use_amp and torch.cuda.is_available()
        
        # Mixed precision scaler [14, 17]
        self.scaler = GradScaler() if self.use_amp else None
        
        # Histórico otimizado
        self.history = {
            'train_loss':, 'train_acc':, 'val_loss':, 'val_acc':,
            'per_class_metrics':, 'learning_rates':
        }
        
        logger.info(f"Trainer configurado com Mixed Precision: {self.use_amp}")

    def train_epoch(self) -> Tuple]:
        """Treina uma época com otimizações"""
        self.model.train()
        running_loss = 0.0
        all_predictions =
        all_labels =
        
        # Progress bar
        pbar = tqdm(self.train_loader, desc="Training", leave=False)
        
        for batch_idx, (images, labels) in enumerate(pbar):
            images, labels = images.to(self.device, non_blocking=True), labels.to(self.device, non_blocking=True)
            
            self.optimizer.zero_grad()
            
            if self.use_amp:
                with autocast(device_type=self.device.type): # Especificar device_type para autocast [14]
                    outputs = self.model(images)
                    loss = self.criterion(outputs, labels)
                
                self.scaler.scale(loss).backward()
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)
                loss.backward()
                self.optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            # Atualizar progress bar
            pbar.set_postfix({'Loss': f'{loss.item():.4f}'})
        
        epoch_loss = running_loss / len(self.train_loader)
        epoch_acc = np.mean(np.array(all_predictions) == np.array(all_labels))
        class_acc = self._calculate_per_class_accuracy(all_labels, all_predictions)
        
        return epoch_loss, epoch_acc, class_acc
    
    def validate_epoch(self) -> Tuple]:
        """Valida uma época com otimizações"""
        self.model.eval()
        running_loss = 0.0
        all_predictions =
        all_labels =
        
        with torch.no_grad():
            pbar = tqdm(self.val_loader, desc="Validating", leave=False)
            for images, labels in pbar:
                images, labels = images.to(self.device, non_blocking=True), labels.to(self.device, non_blocking=True)
                
                if self.use_amp:
                    with autocast(device_type=self.device.type): # Especificar device_type para autocast [14]
                        outputs = self.model(images)
                        loss = self.criterion(outputs, labels)
                else:
                    outputs = self.model(images)
                    loss = self.criterion(outputs, labels)
                
                running_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                all_predictions.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                
                pbar.set_postfix({'Loss': f'{loss.item():.4f}'})
        
        epoch_loss = running_loss / len(self.val_loader)
        epoch_acc = np.mean(np.array(all_predictions) == np.array(all_labels))
        class_acc = self._calculate_per_class_accuracy(all_labels, all_predictions)
        
        return epoch_loss, epoch_acc, class_acc
    
    def _calculate_per_class_accuracy(self, labels: List[int], predictions: List[int]) -> Dict[str, float]:
        """Calcula acurácia por classe otimizada"""
        labels_np = np.array(labels)
        predictions_np = np.array(predictions)
        
        class_acc = {}
        for i, class_name in enumerate(self.class_names):
            class_mask = labels_np == i
            if np.sum(class_mask) > 0: # Garantir que a classe tenha amostras
                class_predictions = predictions_np[class_mask]
                class_labels = labels_np[class_mask]
                class_acc[class_name] = np.mean(class_predictions == class_labels)
            else:
                class_acc[class_name] = 0.0 # Se a classe não tiver amostras, a acurácia é 0
        return class_acc
    
    def train(self, num_epochs: int, early_stopping_patience: int = 10) -> float:
        """Treinamento otimizado com early stopping [20, 12]"""
        best_val_acc = 0.0
        patience_counter = 0
        start_time = time.time()
        
        # Criar diretório para salvar o modelo [13]
        Path("models").mkdir(parents=True, exist_ok=True)
        
        logger.info(f"Iniciando treinamento por {num_epochs} épocas...")
        logger.info("-" * 80)
        
        for epoch in range(num_epochs):
            epoch_start = time.time()
            logger.info(f'Época {epoch+1}/{num_epochs}')
            
            # Treinamento
            train_loss, train_acc, train_class_acc = self.train_epoch()
            
            # Validação
            val_loss, val_acc, val_class_acc = self.validate_epoch()
            
            # Scheduler [8, 21]
            current_lr = self.optimizer.param_groups['lr'] # Acessar LR corretamente
            if self.scheduler:
                if isinstance(self.scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                    self.scheduler.step(val_loss) 
                else:
                    self.scheduler.step()
                
                new_lr = self.optimizer.param_groups['lr']
                if new_lr!= current_lr:
                    logger.info(f'Learning rate: {current_lr:.6f} -> {new_lr:.6f}')
            
            # Salvar histórico
            self.history['train_loss'].append(train_loss)
            self.history['train_acc'].append(train_acc)
            self.history['val_loss'].append(val_loss)
            self.history['val_acc'].append(val_acc)
            self.history['learning_rates'].append(current_lr)
            self.history['per_class_metrics'].append({
                'epoch': epoch,
                'train_class_acc': train_class_acc,
                'val_class_acc': val_class_acc
            })
            
            epoch_time = time.time() - epoch_start
            logger.info(f'Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}')
            logger.info(f'Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}')
            logger.info(f'Tempo da época: {epoch_time:.2f}s')
            
            # Mostrar acurácia por classe
            logger.info("Acurácia por classe (Validação):")
            for class_name, acc in val_class_acc.items():
                logger.info(f"  {class_name}: {acc:.4f}")
            
            # Early stopping com salvamento do melhor modelo [20, 12]
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                patience_counter = 0
                
                # Salvar melhor modelo
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'scheduler_state_dict': self.scheduler.state_dict() if self.scheduler else None,
                    'val_acc': val_acc,
                    'val_loss': val_loss,
                    'class_acc': val_class_acc,
                    'history': self.history
                }, 'models/best_covid_model_optimized.pth')
                
                logger.info(f'✓ Melhor modelo salvo! Val Acc: {val_acc:.4f}')
            else:
                patience_counter += 1
            
            if patience_counter >= early_stopping_patience:
                logger.info(f'Early stopping após {early_stopping_patience} épocas sem melhoria [20, 12]')
                break
            
            logger.info("-" * 80)
        
        total_time = time.time() - start_time
        logger.info(f'Treinamento concluído em {total_time:.2f}s')
        
        return best_val_acc

    def plot_training_history(self, save_path: str = 'plots/training_history_optimized.png'):
        """Plot otimizado do histórico de treinamento"""
        Path("plots").mkdir(parents=True, exist_ok=True) # Criar diretório para plots
        
        fig, axes = plt.subplots(2, 3, figsize=(20, 12))
        epochs = range(1, len(self.history['train_loss']) + 1)
        
        # Loss
        axes.plot(epochs, self.history['train_loss'], 'b-', label='Train Loss', linewidth=2)
        axes.plot(epochs, self.history['val_loss'], 'r-', label='Val Loss', linewidth=2)
        axes.set_title('Model Loss', fontsize=14, fontweight='bold')
        axes.set_xlabel('Epoch')
        axes.set_ylabel('Loss')
        axes.legend()
        axes.grid(True, alpha=0.3)
        
        # Accuracy
        axes.plot(epochs, self.history['train_acc'], 'b-', label='Train Acc', linewidth=2)
        axes.plot(epochs, self.history['val_acc'], 'r-', label='Val Acc', linewidth=2)
        axes.set_title('Model Accuracy', fontsize=14, fontweight='bold')
        axes.set_xlabel('Epoch')
        axes.set_ylabel('Accuracy')
        axes.legend()
        axes.grid(True, alpha=0.3)
        
        # Learning Rate
        axes.plot(epochs, self.history['learning_rates'], 'g-', linewidth=2)
        axes.set_title('Learning Rate', fontsize=14, fontweight='bold')
        axes.set_xlabel('Epoch')
        axes.set_ylabel('Learning Rate')
        axes.set_yscale('log')
        axes.grid(True, alpha=0.3)
        
        # Acurácia por classe
        if self.history['per_class_metrics']:
            for class_name in self.class_names:
                class_accs = [metric['val_class_acc'].get(class_name, 0) 
                             for metric in self.history['per_class_metrics']]
                axes.plot(epochs, class_accs, label=f'{class_name}', 
                               linewidth=2, marker='o', markersize=3)
            
            axes.set_title('Validation Accuracy by Class', fontsize=14, fontweight='bold')
            axes.set_xlabel('Epoch')
            axes.set_ylabel('Class Accuracy')
            axes.legend()
            axes.grid(True, alpha=0.3)
        
        # Overfitting detection
        gap = np.array(self.history['train_acc']) - np.array(self.history['val_acc'])
        axes[1, 1].plot(epochs, gap, 'purple', linewidth=2, label='Train-Val Gap')
        axes[1, 1].axhline(y=0, color='k', linestyle='--', alpha=0.5)
        axes[1, 1].set_title('Overfitting Detection', fontsize=14, fontweight='bold')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Accuracy Gap')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
        
        # Validation Loss vs Accuracy
        axes.[5, 14]scatter(self.history['val_loss'], self.history['val_acc'], 
                          c=epochs, cmap='viridis', alpha=0.7)
        axes.[5, 14]set_title('Val Loss vs Val Accuracy', fontsize=14, fontweight='bold')
        axes.[5, 14]set_xlabel('Validation Loss')
        axes.[5, 14]set_ylabel('Validation Accuracy')
        axes.[5, 14]grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show()

def create_optimized_dataloader(dataset: OptimizedCOVIDDataset, batch_size: int, 
                               is_train: bool = True, num_workers: int = 4) -> Optional:
    """Cria DataLoader otimizado [22, 23, 24, 25]"""
    if not dataset.samples:
        logger.warning(f"Dataset para {dataset.split} está vazio. Não foi possível criar DataLoader.")
        return None
    
    # Otimizar num_workers baseado no sistema
    import multiprocessing
    max_workers = min(num_workers, multiprocessing.cpu_count())
    
    if is_train and len(dataset) > 0:
        # Sampler balanceado otimizado [6, 7, 26, 3, 1]
        labels = [sample[5] for sample in dataset.samples]
        class_counts = Counter(labels)
        
        # Calcular pesos para o sampler, garantindo que classes com 0 amostras não causem erro
        weights =
        for label_idx in labels:
            if class_counts[label_idx] > 0:
                weights.append(1.0 / class_counts[label_idx])
            else:
                weights.append(0.0) # Não amostrar se a classe não tem ocorrências
        
        # Se todos os pesos forem zero (dataset vazio ou classes sem amostras), não criar sampler
        if sum(weights) == 0:
            logger.warning("Todos os pesos do sampler são zero. Não foi possível criar WeightedRandomSampler. Usando shuffle=False.")
            return DataLoader(
                dataset, 
                batch_size=batch_size, 
                shuffle=False, # Não embaralhar se não há pesos válidos
                num_workers=max_workers, 
                pin_memory=torch.cuda.is_available(),
                persistent_workers=max_workers > 0,
                prefetch_factor=2 if max_workers > 0 else 2
            )

        sampler = WeightedRandomSampler(
            weights=torch.DoubleTensor(weights), # Usar DoubleTensor para pesos [7, 3]
            num_samples=len(weights),
            replacement=True
        )
        
        return DataLoader(
            dataset, 
            batch_size=batch_size, 
            sampler=sampler,
            num_workers=max_workers, 
            pin_memory=torch.cuda.is_available(),
            persistent_workers=max_workers > 0,
            prefetch_factor=2 if max_workers > 0 else 2
        )
    else:
        return DataLoader(
            dataset, 
            batch_size=batch_size, 
            shuffle=False,
            num_workers=max_workers, 
            pin_memory=torch.cuda.is_available(),
            persistent_workers=max_workers > 0,
            prefetch_factor=2 if max_workers > 0 else 2
        )

def evaluate_model_optimized(model: nn.Module, test_loader: DataLoader, 
                           class_names: List[str], device: str) -> Dict[str, Any]:
    """Avaliação otimizada do modelo"""
    model.eval()
    all_predictions =
    all_labels =
    all_probs =
    
    logger.info("Avaliando modelo no conjunto de teste...")
    
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Evaluating"):
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            
            if torch.cuda.is_available():
                with autocast(device_type=device.type): # Especificar device_type para autocast [14]
                    outputs = model(images)
            else:
                outputs = model(images)
                
            probs = F.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    # Métricas otimizadas
    overall_accuracy = np.mean(np.array(all_predictions) == np.array(all_labels))
    logger.info(f"Acurácia geral: {overall_accuracy:.4f}")
    
    # Relatório detalhado [24, 21, 19, 2]
    report = classification_report(
        all_labels, all_predictions, 
        target_names=class_names, 
        digits=4,
        output_dict=True,
        zero_division='warn' # Adicionado para lidar com classes sem amostras no relatório [24]
    )
    
    logger.info("\nRelatório de Classificação:")
    logger.info("=" * 60)
    print(classification_report(all_labels, all_predictions, target_names=class_names, digits=4, zero_division='warn'))
    
    # Matriz de confusão [27, 28, 29, 30, 1]
    cm = confusion_matrix(all_labels, all_predictions)
    
    # Visualizações otimizadas
    _plot_confusion_matrix(cm, class_names)
    _plot_roc_curves(all_labels, all_probs, class_names)
    
    return {
        'accuracy': overall_accuracy,
        'predictions': all_predictions,
        'labels': all_labels,
        'probabilities': all_probs,
        'confusion_matrix': cm.tolist(), # Converter para lista para serialização JSON
        'classification_report': report
    }

def _plot_confusion_matrix(cm: np.ndarray, class_names: List[str]):
    """Plot otimizado da matriz de confusão [27, 28, 29, 30]"""
    Path("plots").mkdir(parents=True, exist_ok=True) # Criar diretório para plots
    
    plt.figure(figsize=(15, 6))
    
    # Subplot 1: Matriz de confusão normalizada
    plt.subplot(1, 2, 1)
    # Adicionar um pequeno epsilon para evitar divisão por zero se uma linha for toda zero [27]
    cm_normalized = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-6) 
    sns.heatmap(cm_normalized, annot=True, fmt='.3f', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'label': 'Proporção'})
    plt.title('Matriz de Confusão Normalizada', fontsize=14, fontweight='bold')
    plt.xlabel('Predição')
    plt.ylabel('Real')
    
    # Subplot 2: Matriz de confusão com contagens
    plt.subplot(1, 2, 2)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
                xticklabels=class_names, yticklabels=class_names,
                cbar_kws={'label': 'Contagem'})
    plt.title('Matriz de Confusão (Contagens)', fontsize=14, fontweight='bold')
    plt.xlabel('Predição')
    plt.ylabel('Real')
    
    plt.tight_layout()
    plt.savefig('plots/confusion_matrix_optimized.png', dpi=300, bbox_inches='tight')
    plt.show()

def _plot_roc_curves(labels: List[int], probs: List[List[float]], class_names: List[str]):
    """Plot otimizado das curvas ROC [23, 6, 31, 32, 1]"""
    from sklearn.metrics import roc_curve, auc
    from sklearn.preprocessing import label_binarize
    
    Path("plots").mkdir(parents=True, exist_ok=True) # Criar diretório para plots
    
    # Binarizar labels para multiclass
    # Certificar-se de que todas as classes esperadas estão presentes, mesmo que não em 'labels'
    y_true = label_binarize(labels, classes=list(range(len(class_names))))
    y_scores = np.array(probs)
    
    plt.figure(figsize=(12, 8))
    
    # Cores para cada classe (usar um mapa de cores para mais de 4 classes)
    colors = plt.cm.get_cmap('tab10', len(class_names))
    
    for i, class_name in enumerate(class_names):
        # Verificar se a classe existe na binarização e se tem pelo menos duas classes únicas para calcular ROC
        # (i.e., não é uma classe com todos os rótulos 0 ou todos os rótulos 1)
        if y_true.shape[5] > i and len(np.unique(y_true[:, i])) > 1:
            fpr, tpr, _ = roc_curve(y_true[:, i], y_scores[:, i])
            roc_auc = auc(fpr, tpr)
            
            plt.plot(fpr, tpr, color=colors(i), lw=2, 
                    label=f'{class_name} (AUC = {roc_auc:.3f})')
        else:
            logger.warning(f"Não foi possível plotar ROC para a classe '{class_name}': dados insuficientes (apenas uma classe presente nos rótulos verdadeiros ou classe ausente).")
    
    # Linha diagonal
    plt.plot(, , color='gray', lw=1, linestyle='--', label='Random')
    
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Taxa de Falsos Positivos')
    plt.ylabel('Taxa de Verdadeiros Positivos')
    plt.title('Curvas ROC por Classe', fontsize=14, fontweight='bold')
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('plots/roc_curves_optimized.png', dpi=300, bbox_inches='tight')
    plt.show()

def save_model_info(model: nn.Module, history: Dict, results: Dict, 
                   save_path: str = 'reports/model_info_optimized.json'):
    """Salva informações detalhadas do modelo [13]"""
    
    Path("reports").mkdir(parents=True, exist_ok=True) # Criar diretório para relatórios
    
    # Calcular métricas finais
    best_epoch = np.argmax(history['val_acc'])
    best_val_acc = max(history['val_acc'])
    final_train_acc = history['train_acc'][-1]
    final_val_acc = history['val_acc'][-1]
    
    # Informações do modelo
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    model_info = {
        'model_architecture': model.architecture if hasattr(model, 'architecture') else 'unknown',
        'total_parameters': int(total_params),
        'trainable_parameters': int(trainable_params),
        'training_info': {
            'total_epochs': len(history['train_loss']),
            'best_epoch': int(best_epoch + 1),
            'best_val_accuracy': float(best_val_acc),
            'final_train_accuracy': float(final_train_acc),
            'final_val_accuracy': float(final_val_acc),
            'final_train_loss': float(history['train_loss'][-1]),
            'final_val_loss': float(history['val_loss'][-1])
        },
        'test_results': {
            'test_accuracy': float(results['accuracy']),
            'classification_report': results['classification_report']
        },
        'training_history': {
            'train_loss': [float(x) for x in history['train_loss']],
            'val_loss': [float(x) for x in history['val_loss']],
            'train_acc': [float(x) for x in history['train_acc']],
            'val_acc': [float(x) for x in history['val_acc']],
            'learning_rates': [float(x) for x in history['learning_rates']]
        }
    }
    
    with open(save_path, 'w') as f:
        json.dump(model_info, f, indent=2)
    
    logger.info(f"Informações do modelo salvas em: {save_path}")
    return model_info

def load_best_model(model: nn.Module, checkpoint_path: str = 'models/best_covid_model_optimized.pth'):
    """Carrega o melhor modelo salvo"""
    if os.path.exists(checkpoint_path):
        logger.info(f"Carregando modelo de: {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        
        logger.info(f"Modelo carregado da época {checkpoint['epoch']} com Val Acc: {checkpoint['val_acc']:.4f}")
        return checkpoint
    else:
        logger.warning(f"Checkpoint não encontrado: {checkpoint_path}. O modelo não será carregado.")
        return None

def main():
    """Função principal otimizada"""
    logger.info("Iniciando classificador COVID-19 otimizado")
    logger.info("=" * 80)
    
    # Configurações
    DATASET_ROOT = "./covid_dataset"  # Ajuste o caminho conforme necessário
    BATCH_SIZE = 32
    LEARNING_RATE = 1e-4
    NUM_EPOCHS = 50
    EARLY_STOPPING_PATIENCE = 10
    ARCHITECTURE = 'efficientnet_b3'  # ou 'resnet50'
    
    try:
        # 1. Preparar transformações
        train_transforms = get_optimized_transforms('train')
        val_transforms = get_optimized_transforms('val')
        
        # 2. Carregar datasets
        logger.info("Carregando datasets...")
        train_dataset = OptimizedCOVIDDataset(
            DATASET_ROOT, 'train', 
            transform=train_transforms,
            cache_images=False  # True para datasets pequenos
        )
        
        val_dataset = OptimizedCOVIDDataset(
            DATASET_ROOT, 'val', 
            transform=val_transforms,
            cache_images=False
        )
        
        test_dataset = OptimizedCOVIDDataset(
            DATASET_ROOT, 'test', 
            transform=val_transforms,
            cache_images=False
        )
        
        # Verificar se os datasets foram carregados com sucesso
        if len(train_dataset) == 0:
            logger.error("Dataset de treino vazio! Verifique o caminho e a estrutura do dataset. Abortando.")
            return # Sair se o dataset de treino estiver vazio
        if len(val_dataset) == 0:
            logger.warning("Dataset de validação vazio! O treinamento pode não ser avaliado corretamente.")
        if len(test_dataset) == 0:
            logger.warning("Dataset de teste vazio! A avaliação final não será realizada.")
        
        logger.info(f"Dataset carregado - Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")
        
        # 3. Criar data loaders
        train_loader = create_optimized_dataloader(train_dataset, BATCH_SIZE, is_train=True)
        val_loader = create_optimized_dataloader(val_dataset, BATCH_SIZE, is_train=False)
        test_loader = create_optimized_dataloader(test_dataset, BATCH_SIZE, is_train=False)
        
        if train_loader is None:
            logger.error("Não foi possível criar o train_loader! Verifique o dataset. Abortando.")
            return # Sair se o train_loader não puder ser criado
        
        # 4. Criar modelo
        logger.info(f"Criando modelo com arquitetura: {ARCHITECTURE}")
        model = OptimizedCOVIDClassifier(
            num_classes=len(train_dataset.class_names), # Usar o número de classes do dataset
            pretrained=True,
            dropout_rate=0.3,
            architecture=ARCHITECTURE
        ).to(device)
        
        # Informações do modelo
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        logger.info(f"Parâmetros totais: {total_params:,}")
        logger.info(f"Parâmetros treináveis: {trainable_params:,}")
        
        # 5. Configurar otimização
        class_weights = train_dataset.get_class_weights().to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights) # [20, 33, 8, 27]
        
        optimizer = optim.AdamW( # [34, 7]
            model.parameters(), 
            lr=LEARNING_RATE,
            weight_decay=1e-4, # 
            eps=1e-8
        )
        
        scheduler = optim.lr_scheduler.ReduceLROnPlateau( # [8, 21]
            optimizer, 
            mode='min',
            patience=5,
            factor=0.5,
            min_lr=1e-7,
            verbose=True
        )
        
        # 6. Criar trainer
        trainer = OptimizedModelTrainer(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            device=device,
            class_names=train_dataset.class_names,
            use_amp=True
        )
        
        # 7. Treinar modelo
        best_val_acc = trainer.train(NUM_EPOCHS, EARLY_STOPPING_PATIENCE)
        
        # 8. Plotar histórico de treinamento
        trainer.plot_training_history()
        
        # 9. Carregar melhor modelo e avaliar
        checkpoint = load_best_model(model)
        
        if test_loader and len(test_dataset) > 0:
            logger.info("Avaliando no conjunto de teste...")
            test_results = evaluate_model_optimized(
                model, test_loader, train_dataset.class_names, device
            )
            
            # 10. Salvar informações do modelo
            model_info = save_model_info(model, trainer.history, test_results)
            
            logger.info("\n" + "=" * 80)
            logger.info("RESULTADOS FINAIS")
            logger.info("=" * 80)
            logger.info(f"Melhor acurácia de validação: {best_val_acc:.4f}")
            logger.info(f"Acurácia de teste: {test_results['accuracy']:.4f}")
            logger.info(f"Arquitetura: {ARCHITECTURE}")
            logger.info(f"Parâmetros treináveis: {trainable_params:,}")
            
        else:
            logger.warning("Conjunto de teste não disponível ou vazio para avaliação final.")
            
    except Exception as e:
        logger.error(f"Erro crítico durante a execução principal: {e}", exc_info=True)
        # Re-raise para que o erro seja visível no console/logs externos
        raise

if __name__ == "__main__":
    # Para rodar o treinamento e avaliação:
    main()

    # --- Funções de exemplo para uso pós-treinamento ---
    # Para usar as funções abaixo, certifique-se de que o `main()` foi executado
    # e salvou um modelo em 'models/best_covid_model_optimized.pth'.

    # Exemplo de uso da função de predição para uma única imagem
    # Descomente para testar:
    # print("\n" + "="*80)
    # logger.info("Executando exemplo de predição de imagem única...")
    # example_prediction()

    # Exemplo de análise de classificações incorretas
    # Descomente para testar:
    # print("\n" + "="*80)
    # logger.info("Executando análise de classificações incorretas...")
    # # Você precisará recarregar o modelo e o test_loader se o main() já terminou
    # # class_names = ['covid19', 'normal', 'pneumonia_bacterial', 'pneumonia_viral']
    # # model_for_analysis = OptimizedCOVIDClassifier(num_classes=len(class_names), architecture='efficientnet_b3').to(device)
    # # load_best_model(model_for_analysis)
    # # # Recriar test_dataset e test_loader se necessário, pois o main() pode ter terminado
    # # # test_dataset_for_analysis = OptimizedCOVIDDataset("./covid_dataset", 'test', transform=get_optimized_transforms('val'))
    # # # test_loader_for_analysis = create_optimized_dataloader(test_dataset_for_analysis, 32, is_train=False)
    # # # analyze_misclassifications(model_for_analysis, test_loader_for_analysis, class_names, device, num_examples=5)

# Função auxiliar para inferência
def predict_single_image(model: nn.Module, image_path: str, 
                        class_names: List[str], device: str,
                        transform=None) -> Dict[str, Any]:
    """Predição para uma única imagem"""
    model.eval()
    
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Imagem não encontrada no caminho: {image_path}")

    # Carregar e transformar imagem
    image = Image.open(image_path).convert("RGB")
    
    if transform is None:
        transform = get_optimized_transforms('val')
    
    # Aplicar transformações
    if _ALBUMENTATIONS_AVAILABLE and isinstance(transform, A.Compose):  # Albumentations [9, 10, 28, 35]
        image_np = np.array(image)
        transformed = transform(image=image_np)
        image_tensor = transformed['image'].unsqueeze(0)
    else:  # torchvision
        image_tensor = transform(image).unsqueeze(0)
    
    image_tensor = image_tensor.to(device)
    
    with torch.no_grad():
        if torch.cuda.is_available():
            with autocast(device_type=device.type): # Especificar device_type para autocast [14]
                outputs = model(image_tensor)
        else:
            outputs = model(image_tensor)
        
        probabilities = F.softmax(outputs, dim=1)
        predicted_class = torch.argmax(outputs, dim=1)
    
    # Preparar resultados
    probs_dict = {class_names[i]: float(probabilities[i]) # Acessar probabilities para batch size 1
                  for i in range(len(class_names))}
    
    return {
        'predicted_class': class_names[predicted_class.item()],
        'confidence': float(probabilities[predicted_class].item()),
        'all_probabilities': probs_dict,
        'raw_output': outputs.cpu().numpy().tolist() # Converter para lista para serialização
    }

# Exemplo de uso da função de predição
def example_prediction():
    """Exemplo de como usar a predição para uma imagem"""
    
    # Carregar modelo
    # Certifique-se de que o número de classes aqui corresponde ao modelo treinado
    class_names = ['covid19', 'normal', 'pneumonia_bacterial', 'pneumonia_viral']
    model = OptimizedCOVIDClassifier(num_classes=len(class_names), architecture='efficientnet_b3').to(device)
    checkpoint = load_best_model(model)
    
    if checkpoint is None:
        logger.error("Modelo não encontrado! Execute o treinamento primeiro para gerar 'models/best_covid_model_optimized.pth'.")
        return
    
    # Caminho da imagem (ajuste conforme necessário)
    # Crie um arquivo dummy para teste se não tiver um dataset real
    dummy_image_path = "exemplo_imagem.jpg"
    if not os.path.exists(dummy_image_path):
        logger.warning(f"Criando imagem dummy em {dummy_image_path} para exemplo de predição. Esta imagem será vermelha.")
        try:
            Image.new('RGB', (224, 224), color = 'red').save(dummy_image_path)
        except Exception as e:
            logger.error(f"Não foi possível criar imagem dummy em {dummy_image_path}: {e}")
            return
    
    try:
        # Fazer predição
        result = predict_single_image(model, dummy_image_path, class_names, device)
        
        print("\n" + "="*50)
        print("RESULTADO DA PREDIÇÃO")
        print("="*50)
        print(f"Classe predita: {result['predicted_class']}")
        print(f"Confiança: {result['confidence']:.4f}")
        print("\nProbabilidades por classe:")
        for class_name, prob in result['all_probabilities'].items():
            print(f"  {class_name}: {prob:.4f}")
            
    except FileNotFoundError as e:
        logger.error(f"Erro: {e}. Certifique-se de que o caminho da imagem está correto.")
    except Exception as e:
        logger.error(f"Erro na predição: {e}", exc_info=True)

# Função para análise de erros
def analyze_misclassifications(model: nn.Module, test_loader: DataLoader, 
                             class_names: List[str], device: str, 
                             num_examples: int = 10):
    """Analisa classificações incorretas para debugging [31, 35, 36]"""
    model.eval()
    misclassified =
    
    if test_loader is None or len(test_loader.dataset) == 0:
        logger.warning("Test loader não disponível ou vazio para análise de classificações incorretas.")
        return

    logger.info("Analisando classificações incorretas...")
    
    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(tqdm(test_loader, desc="Analyzing misclassifications")):
            images, labels = images.to(device), labels.to(device)
            
            if torch.cuda.is_available():
                with autocast(device_type=device.type): # Especificar device_type para autocast [14]
                    outputs = model(images)
            else:
                outputs = model(images)
            
            _, predicted = torch.max(outputs, 1)
            incorrect_mask = predicted!= labels
            
            if torch.any(incorrect_mask):
                incorrect_indices = torch.where(incorrect_mask) # Acessar o tensor de índices
                
                for idx in incorrect_indices:
                    if len(misclassified) >= num_examples:
                        break
                        
                    probs = F.softmax(outputs[idx], dim=0)
                    misclassified.append({
                        'true_label': class_names[labels[idx].item()],
                        'predicted_label': class_names[predicted[idx].item()],
                        'confidence': probs[predicted[idx]].item(),
                        'all_probs': {class_names[i]: probs[i].item() 
                                    for i in range(len(class_names))},
                        'image': images[idx].cpu() # Armazenar tensor da imagem para possível visualização
                    })
            
            if len(misclassified) >= num_examples:
                break
    
    # Exibir análise
    if misclassified:
        logger.info(f"\nAnálise de {len(misclassified)} classificações incorretas:")
        logger.info("-" * 60)
        
        for i, example in enumerate(misclassified):
            logger.info(f"\nExemplo {i+1}:")
            logger.info(f"  Real: {example['true_label']}")
            logger.info(f"  Predito: {example['predicted_label']} (confiança: {example['confidence']:.3f})")
            logger.info("  Probabilidades:")
            for class_name, prob in example['all_probs'].items():
                marker = "←" if class_name == example['predicted_label'] else ""
                logger.info(f"    {class_name}: {prob:.3f} {marker}")
    else:
        logger.info("Nenhuma classificação incorreta encontrada no número de exemplos solicitados.")
    
    return misclassified

SyntaxError: unmatched ']' (2131571189.py, line 240)